# Etap 1 - Pozyskanie danych z cBioPortal API

**Projekt:** Analiza przeżywalności pacjentów z glejakami w kohorcie TCGA  
**Autor:** Anna Zimniewicz  
**Data:** 18 maja 2026

## Cel notebooka

Pobranie surowych danych klinicznych i molekularnych dla dwóch zbiorów TCGA z cBioPortal:
- **TCGA-GBM** (Glioblastoma Multiforme, Firehose Legacy) - 619 próbek
- **TCGA-LGG** (Brain Lower Grade Glioma, Firehose Legacy) - 530 próbek

Dane zostaną zapisane jako surowe pliki CSV w folderze `data/raw/`.

## Plan

1. Konfiguracja: import bibliotek, ustawienie bazowego URL API
2. Sprawdzenie statusu serwera cBioPortal (`/api/health`)
3. Pobranie listy wszystkich studies i znalezienie `studyId` dla naszych dwóch kohort
4. Pobranie danych klinicznych dla obu studies
5. Pobranie mutacji dla genów IDH1, IDH2
6. Zapis surowych danych do `../data/raw/`

## Źródło danych

- cBioPortal: https://www.cbioportal.org
- Dokumentacja API: https://www.cbioportal.org/api/swagger-ui/index.html

---

## Sekcja 1 — Konfiguracja

Importy bibliotek, definicja stałych projektu (URL bazowy API, identyfikatory studies, ścieżka do surowych danych).

In [4]:
# === KONFIGURACJA ===

# Bazowy URL cBioPortal API
# Wszystkie endpointy budujemy doklejając kolejne fragmenty do tego URL-a
BASE_URL = "https://www.cbioportal.org/api"

# Identyfikatory dwóch studies, które nas interesują
# (te ID znajdziemy za chwilę przez API - na razie wpisujemy je z głowy na podstawie znajomości cBioPortal)
STUDY_IDS = {
    "GBM": "gbm_tcga",       # Glioblastoma Multiforme (TCGA, Firehose Legacy)
    "LGG": "lgg_tcga",       # Brain Lower Grade Glioma (TCGA, Firehose Legacy)
}

# Ścieżka do folderu na surowe dane
# Path("..") oznacza "folder wyżej" - bo notebook jest w notebooks/, a data jest w głównym folderze projektu
RAW_DATA_DIR = Path("..") / "data" / "raw"

# Upewniamy się, że folder istnieje (jeśli nie - utwórz)
# parents=True: utwórz też foldery nadrzędne, jeśli nie istnieją
# exist_ok=True: nie rzucaj błędu, jeśli folder już jest
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Data pobrania - przyda się do nazw plików i dokumentacji
DOWNLOAD_DATE = datetime.now().strftime("%Y-%m-%d")

# Wypisujemy konfigurację dla kontroli
print(f"Base URL: {BASE_URL}")
print(f"Studies: {STUDY_IDS}")
print(f"Folder docelowy: {RAW_DATA_DIR.resolve()}")  # .resolve() pokazuje pełną ścieżkę
print(f"Data pobrania: {DOWNLOAD_DATE}")

Base URL: https://www.cbioportal.org/api
Studies: {'GBM': 'gbm_tcga', 'LGG': 'lgg_tcga'}
Folder docelowy: C:\Users\anazi\Desktop\praca_dyplomowa\data\raw
Data pobrania: 2026-05-27


In [3]:
# Import bibliotek potrzebnych do pobierania i obróbki danych
import requests              # wysyłanie zapytań HTTP do API
import pandas as pd          # praca z danymi tabelarycznymi 
from pathlib import Path     # nowoczesna obsługa ścieżek plików 
from datetime import datetime  # do oznaczania daty pobrania danych

# Sprawdzamy, czy biblioteki się załadowały
print("Wszystkie biblioteki zaimportowane poprawnie.")
print(f"pandas version: {pd.__version__}")
print(f"requests version: {requests.__version__}")

Wszystkie biblioteki zaimportowane poprawnie.
pandas version: 3.0.2
requests version: 2.33.1


---

## Sekcja 2 — Weryfikacja serwera i studies

Sprawdzamy, czy:
- serwer cBioPortal odpowiada poprawnie (`/api/health`),
- studies `gbm_tcga` i `lgg_tcga` istnieją i są to faktycznie TCGA Firehose Legacy,
- liczby pacjentów odpowiadają wartościom widocznym na stronie webowej cBioPortal.

**Uwaga diagnostyczna:** podczas weryfikacji wykryto błąd w polu `allSampleCount` w metadanych API (zwraca `1` zamiast prawidłowej liczby próbek). Bug nie wpływa na dalszą pracę — realną wielkość kohorty ustalamy na podstawie endpointu `/patients`.

In [5]:
# === KROK 1: Sprawdzenie statusu serwera cBioPortal ===

# Budujemy URL endpointu przez sklejenie bazowego URL-a z konkretną ścieżką
health_url = f"{BASE_URL}/health"

# Wysyłamy zapytanie GET
# requests.get() zwraca obiekt Response zawierający odpowiedź serwera
response = requests.get(health_url)

# Sprawdzamy, czy zapytanie się powiodło
# raise_for_status() rzuca wyjątek, jeśli status_code nie jest w zakresie 200-299
response.raise_for_status()

# Wyświetlamy informacje diagnostyczne
print(f"URL zapytania: {health_url}")
print(f"Status code: {response.status_code}")
print(f"Czas odpowiedzi: {response.elapsed.total_seconds()} s")
print(f"Treść odpowiedzi: {response.text}")

URL zapytania: https://www.cbioportal.org/api/health
Status code: 200
Czas odpowiedzi: 0.656849 s
Treść odpowiedzi: {"status":"UP"}


In [6]:
# === KROK 2: Pobranie listy wszystkich studies w cBioPortal ===

# Endpoint /studies zwraca listę wszystkich dostępnych badań w cBioPortal
# Dodajemy parametr projection=SUMMARY - dostaniemy skrócone metadane (bez wszystkich detali)

studies_url = f"{BASE_URL}/studies" 
params = {
    "projection" : "SUMMARY", 
    "pageSize" : 10_000_000,
    "pageNumber" : 0,
    "direction" : "ASC",
}

response = requests.get(studies_url, params=params)
response.raise_for_status()

studies_data = response.json()

print(f"Liczba studies w cBioPortal: {len(studies_data)}")
print(f"Typ obiektu: {type(studies_data)}")
print(f"Typ pierwszego elementu: {type(studies_data[0])}")
print(f"\nKlucze w pierwszym elemencie:")
print(list(studies_data[0].keys()))



Liczba studies w cBioPortal: 535
Typ obiektu: <class 'list'>
Typ pierwszego elementu: <class 'dict'>

Klucze w pierwszym elemencie:
['name', 'description', 'publicStudy', 'groups', 'status', 'importDate', 'allSampleCount', 'readPermission', 'resourceCounts', 'studyId', 'cancerTypeId', 'referenceGenome']


In [7]:
# === KROK 3: Konwersja na DataFrame i filtrowanie do naszych studies ===
# pandas potrafi zamienić listę słowników bezpośrednio w DataFrame
# Każdy słownik → jeden wiersz, klucze słownika → nazwy kolumn

studies_df = pd.DataFrame(studies_data)

print(f"Wymiary tabeli: {studies_df.shape}") #.shape zwraca (liczba_wierszy, liczba_kolumn)
print(f"Kolumny: {list(studies_df.columns)}")

print("\nTrzy pierwsze studies (wybrane kolumny):")
display(studies_df[["studyId", "name", "cancerTypeId", "allSampleCount"]].head(3)) #.head(3) zwraca 3 pierwsze wiersze 


Wymiary tabeli: (535, 14)
Kolumny: ['name', 'description', 'publicStudy', 'groups', 'status', 'importDate', 'allSampleCount', 'readPermission', 'resourceCounts', 'studyId', 'cancerTypeId', 'referenceGenome', 'pmid', 'citation']

Trzy pierwsze studies (wybrane kolumny):


,studyId,name,cancerTypeId,allSampleCount
0,breast_cptac_gdc,"Breast Cancer (CPTAC GDC, 2025)",breast,1
1,glioma_mskcc_2019,"Glioma (MSK, Clin Cancer Res 2019)",difg,1
2,os_target_gdc,"Osteosarcoma (TARGET GDC, 2025)",os,1


Wyjaśnienie: te trzy wiersze to nowo dodane studies w stanie roboczym ("placeholdery" z 1 testową próbką). Sortowanie alfabetyczne wrzuciło je na początek. Nasze GBM (619 próbek) i LGG (530 próbek) są gdzieś dalej w tabeli.

In [8]:
# === KROK 4: Filtrowanie do naszych dwóch studies (GBM + LGG Firehose Legacy) ===

# Lista ID, których szukamy (bierzemy je ze słownika STUDY_IDS)
target_ids = list(STUDY_IDS.values())  # ['gbm_tcga', 'lgg_tcga'] #.values zwraca wartości słownika, list zamienia do na listę
print(f"Szukamy studies o ID: {target_ids}")

# Filtrowanie DataFrame metodą .isin()
# Tworzymy warunek logiczny: które wiersze mają studyId w naszej liście?
#Weź kolumnę studyId z DataFrame. 
#Dla każdego wiersza sprawdź, czy wartość w tej kolumnie znajduje się w liście target_ids. 
#Zwróć kolumnę z True/False.
mask = studies_df["studyId"].isin(target_ids)

our_studies = studies_df[mask]

print(f"\nZnaleziono {len(our_studies)} pasujących studies:")
display(our_studies[["studyId", "name", "cancerTypeId", "allSampleCount", "referenceGenome"]])

Szukamy studies o ID: ['gbm_tcga', 'lgg_tcga']

Znaleziono 2 pasujących studies:


,studyId,name,cancerTypeId,allSampleCount,referenceGenome
95,gbm_tcga,"Glioblastoma Multiforme (TCGA, Firehose Legacy)",difg,1,hg19
227,lgg_tcga,"Brain Lower Grade Glioma (TCGA, Firehose Legacy)",difg,1,hg19


allSampleCount się nie zgadza 

## Sekcja 3 — Eksploracja kohorty

Pobieramy szczegółowe metadane studies (`projection=DETAILED`) oraz listę pacjentów dla obu kohort. Ustalamy realną wielkość kohorty:

- **TCGA-GBM:** 606 pacjentów
- **TCGA-LGG:** 516 pacjentów
- **Razem:** 1122 pacjentów

To są dane, na których zostanie oparta dalsza analiza.

In [9]:
# === DIAGNOSTYKA: Sprawdzenie metadanych dla naszych studies z DETAILED projection ===

# Powtarzamy zapytanie, ale tym razem z projection=DETAILED
# Powinniśmy dostać pełne metadane, w tym prawdziwy allSampleCount
params_detailed = {
    "projection": "DETAILED",
    "pageSize": 10_000_000,
    "pageNumber": 0,
    "direction": "ASC",
}

response = requests.get(f"{BASE_URL}/studies", params=params_detailed)
response.raise_for_status()
studies_detailed = pd.DataFrame(response.json())

print(f"Liczba kolumn z DETAILED: {len(studies_detailed.columns)}")
print(f"Kolumny: {list(studies_detailed.columns)}")

# Filtrujemy do naszych dwóch studies
our_studies_detailed = studies_detailed[studies_detailed["studyId"].isin(target_ids)]

print("\nNasze studies z DETAILED projection:")
display(our_studies_detailed)

Liczba kolumn z DETAILED: 27
Kolumny: ['name', 'description', 'publicStudy', 'groups', 'status', 'importDate', 'allSampleCount', 'sequencedSampleCount', 'cnaSampleCount', 'mrnaRnaSeqSampleCount', 'mrnaRnaSeqV2SampleCount', 'mrnaMicroarraySampleCount', 'miRnaSampleCount', 'methylationHm27SampleCount', 'rppaSampleCount', 'massSpectrometrySampleCount', 'completeSampleCount', 'readPermission', 'treatmentCount', 'structuralVariantCount', 'resourceCounts', 'studyId', 'cancerTypeId', 'cancerType', 'referenceGenome', 'pmid', 'citation']

Nasze studies z DETAILED projection:


,name,description,publicStudy,groups,status,importDate,allSampleCount,sequencedSampleCount,cnaSampleCount,mrnaRnaSeqSampleCount,...,readPermission,treatmentCount,structuralVariantCount,resourceCounts,studyId,cancerTypeId,cancerType,referenceGenome,pmid,citation
95,"Glioblastoma Multiforme (TCGA, Firehose Legacy)",TCGA Glioblastoma Multiforme. Source data from...,True,PUBLIC,0,2026-01-07 12:59:53,1,290,577,0,...,True,0,0,[],gbm_tcga,difg,"{'name': 'Diffuse Glioma', 'dedicatedColor': '...",hg19,NaN,NaN
227,"Brain Lower Grade Glioma (TCGA, Firehose Legacy)",TCGA Brain Lower Grade Glioma. Source data fro...,True,PUBLIC,0,2026-01-07 15:31:49,1,286,513,0,...,True,0,0,[],lgg_tcga,difg,"{'name': 'Diffuse Glioma', 'dedicatedColor': '...",hg19,NaN,NaN


## ⚠️ Uwaga diagnostyczna - bug w cBioPortal API

Endpoint `/studies` zwraca `allSampleCount: 1` dla obu naszych studies (`gbm_tcga`, `lgg_tcga`), 
zarówno z `projection=SUMMARY` jak i `DETAILED`. To jest **bug w metadanych cBioPortal** - na stronie 
webowej widać poprawne liczby (619 dla GBM, 530 dla LGG), ale licznik w API się rozjechał.

**Inne pola w DETAILED zawierają prawdziwe liczby per typ danych:**
- `sequencedSampleCount`: 290 (GBM), 286 (LGG) - próbki z danymi mutacji
- `cnaSampleCount`: 577 (GBM), 513 (LGG) - próbki z danymi CNA

**Realna wielkość kohorty zostanie ustalona** na podstawie liczby unikalnych pacjentów 
w pobranych danych klinicznych (krok 5).

In [10]:
# === KROK 5: Pobranie listy pacjentów dla obu studies ===

# Endpoint /studies/{studyId}/patients zwraca listę wszystkich pacjentów w danym studium
# Robimy to w pętli dla obu naszych studies (GBM i LGG)

patients_per_study = {}  # tu będziemy zbierać wyniki

for label, study_id in STUDY_IDS.items():
    # label = "GBM" lub "LGG", study_id = "gbm_tcga" lub "lgg_tcga"
    
    # Budujemy URL z konkretnym study_id
    url = f"{BASE_URL}/studies/{study_id}/patients"
    
    # Wysyłamy zapytanie z dużym pageSize, żeby dostać wszystko naraz
    params = {"pageSize": 100_000, "pageNumber": 0}
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    # Parsujemy odpowiedź i zamieniamy na DataFrame
    patients = pd.DataFrame(response.json())
    
    # Zapisujemy do słownika pod kluczem "GBM" / "LGG"
    patients_per_study[label] = patients
    
    print(f"{label} ({study_id}): {len(patients)} pacjentów")
    
print(f"\nŁącznie pacjentów: {sum(len(df) for df in patients_per_study.values())}")
print("\nKolumny w DataFrame pacjentów GBM:")
print(list(patients_per_study["GBM"].columns))
print("\nPierwsze 3 wiersze GBM:")
display(patients_per_study["GBM"].head(3))

GBM (gbm_tcga): 606 pacjentów
LGG (lgg_tcga): 516 pacjentów

Łącznie pacjentów: 1122

Kolumny w DataFrame pacjentów GBM:
['uniquePatientKey', 'patientId', 'studyId']

Pierwsze 3 wiersze GBM:


,uniquePatientKey,patientId,studyId
0,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga
1,VENHQS0wMi0wMDAzOmdibV90Y2dh,TCGA-02-0003,gbm_tcga
2,VENHQS0wMi0wMDA2OmdibV90Y2dh,TCGA-02-0006,gbm_tcga


In [11]:
# === KROK 6: Pobranie danych klinicznych w formacie LONG ===

# Słownik, w którym zbierzemy dane kliniczne per studium
clinical_long_per_study = {}

for label, study_id in STUDY_IDS.items():
    # Endpoint zwraca wszystkie dane kliniczne dla pacjentów w studium
    url = f"{BASE_URL}/studies/{study_id}/clinical-data"
    
    # clinicalDataType=PATIENT - chcemy atrybuty pacjenta (nie próbki)
    # projection=SUMMARY - mniej pól, ale wszystko, czego nam trzeba
    params = {
        "clinicalDataType": "PATIENT",
        "projection": "SUMMARY",
        "pageSize": 1_000_000,
        "pageNumber": 0,
    }
    
    print(f"Pobieram dane kliniczne dla {label} ({study_id})...")
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    # Zamieniamy odpowiedź na DataFrame
    clinical_long = pd.DataFrame(response.json())
    clinical_long_per_study[label] = clinical_long
    
    print(f"  Pobrano {len(clinical_long):,} wierszy (long format)")
    print(f"  Liczba unikalnych pacjentów: {clinical_long['patientId'].nunique()}")
    print(f"  Liczba unikalnych atrybutów: {clinical_long['clinicalAttributeId'].nunique()}")
    print()

# Pokaż pierwsze 10 wierszy GBM dla wglądu w surową strukturę long
print("Pierwsze 10 wierszy danych klinicznych GBM (format LONG):")
display(clinical_long_per_study["GBM"].head(10))

Pobieram dane kliniczne dla GBM (gbm_tcga)...
  Pobrano 14,830 wierszy (long format)
  Liczba unikalnych pacjentów: 606
  Liczba unikalnych atrybutów: 34

Pobieram dane kliniczne dla LGG (lgg_tcga)...
  Pobrano 25,322 wierszy (long format)
  Liczba unikalnych pacjentów: 516
  Liczba unikalnych atrybutów: 65

Pierwsze 10 wierszy danych klinicznych GBM (format LONG):


,uniquePatientKey,patientId,studyId,clinicalAttributeId,value
0,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,AGE,44
1,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS,0
2,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,DFS_MONTHS,4.5
3,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,DFS_STATUS,1:Recurred/Progressed
4,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,ETHNICITY,NOT HISPANIC OR LATINO
5,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,FORM_COMPLETION_DATE,12/16/08
6,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,HISTOLOGICAL_DIAGNOSIS,Untreated primary (de novo) GBM
7,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,HISTORY_LGG_DX_OF_BRAIN_TISSUE,NO
8,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,HISTORY_NEOADJUVANT_TRTYN,Yes
9,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,ICD_10,C71.9


In [12]:
# === KROK 7: Eksploracja - jakie atrybuty kliniczne są dostępne ===

# Pobieramy unikalne nazwy atrybutów dla każdego studium
attrs_gbm = set(clinical_long_per_study["GBM"]["clinicalAttributeId"].unique())
attrs_lgg = set(clinical_long_per_study["LGG"]["clinicalAttributeId"].unique())

# Atrybuty wspólne (są w obu studies)
common_attrs = attrs_gbm & attrs_lgg  # operator & to "część wspólna" zbiorów

# Atrybuty tylko w jednym ze studies
only_gbm = attrs_gbm - attrs_lgg
only_lgg = attrs_lgg - attrs_gbm

print(f"Atrybuty wspólne dla obu studies: {len(common_attrs)}")
print(f"Atrybuty tylko w GBM: {len(only_gbm)}")
print(f"Atrybuty tylko w LGG: {len(only_lgg)}")

print("\n=== Atrybuty wspólne dla obu studies (alfabetycznie) ===")
for attr in sorted(common_attrs):
    print(f"  {attr}")

print("\n=== Atrybuty tylko w LGG ===")
for attr in sorted(only_lgg):
    print(f"  {attr}")

print("\n=== Atrybuty tylko w GBM ===")
for attr in sorted(only_gbm):
    print(f"  {attr}")

Atrybuty wspólne dla obu studies: 31
Atrybuty tylko w GBM: 3
Atrybuty tylko w LGG: 34

=== Atrybuty wspólne dla obu studies (alfabetycznie) ===
  AGE
  DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS
  DFS_MONTHS
  DFS_STATUS
  ECOG_SCORE
  ETHNICITY
  FORM_COMPLETION_DATE
  HISTOLOGICAL_DIAGNOSIS
  HISTORY_NEOADJUVANT_TRTYN
  HISTORY_OTHER_MALIGNANCY
  ICD_10
  ICD_O_3_HISTOLOGY
  ICD_O_3_SITE
  INFORMED_CONSENT_VERIFIED
  INITIAL_PATHOLOGIC_DX_YEAR
  KARNOFSKY_PERFORMANCE_SCORE
  NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT
  OS_MONTHS
  OS_STATUS
  OTHER_PATIENT_ID
  PERFORMANCE_STATUS_TIMING
  PROSPECTIVE_COLLECTION
  RACE
  RADIATION_TREATMENT_ADJUVANT
  RETROSPECTIVE_COLLECTION
  SAMPLE_COUNT
  SEX
  SITE_OF_TUMOR_TISSUE
  TISSUE_SOURCE_SITE
  TREATMENT_OUTCOME_FIRST_COURSE
  TUMOR_STATUS

=== Atrybuty tylko w LGG ===
  ANIMAL_INSECT_ALLERGY_AGE
  ANIMAL_INSECT_ALLERGY_HIST
  ASTHMA_ECZEMA_ALLERGY_FIRST_DIAGNOSIS
  ASTHMA_HISTORY
  ECZEMA_HISTORY
  FAMILY_HISTORY_OF_CANCER
  FAMILY_HISTORY_OF_PR

In [13]:
# === DIAGNOZA: sprawdzenie atrybutów na poziomie SAMPLE (próbki) ===

# Powtarzamy zapytanie, ale tym razem dla danych na poziomie próbki, nie pacjenta
url = f"{BASE_URL}/studies/gbm_tcga/clinical-data"
params = {
    "clinicalDataType": "SAMPLE",   # ← TO jest jedyna zmiana
    "projection": "SUMMARY",
    "pageSize": 1_000_000,
    "pageNumber": 0,
}

response = requests.get(url, params=params)
response.raise_for_status()
sample_clinical_gbm = pd.DataFrame(response.json())

print(f"Pobrano {len(sample_clinical_gbm):,} wierszy danych SAMPLE dla GBM")
print(f"Liczba unikalnych atrybutów SAMPLE: {sample_clinical_gbm['clinicalAttributeId'].nunique()}")

# Pokażemy wszystkie atrybuty SAMPLE
sample_attrs_gbm = sorted(sample_clinical_gbm["clinicalAttributeId"].unique())
print("\n=== Wszystkie atrybuty na poziomie SAMPLE dla GBM ===")
for attr in sample_attrs_gbm:
    # Podświetlamy te, które brzmią dla nas istotnie
    is_key = any(keyword in attr.upper() for keyword in ["MGMT", "IDH", "METHYL", "MUTATION"])
    marker = "  ⭐ " if is_key else "     "
    print(f"{marker}{attr}")

Pobrano 9,506 wierszy danych SAMPLE dla GBM
Liczba unikalnych atrybutów SAMPLE: 20

=== Wszystkie atrybuty na poziomie SAMPLE dla GBM ===
     CANCER_TYPE
     CANCER_TYPE_DETAILED
     DAYS_TO_COLLECTION
     FRACTION_GENOME_ALTERED
     IS_FFPE
     LONGEST_DIMENSION
  ⭐ MUTATION_COUNT
     OCT_EMBEDDED
     ONCOTREE_CODE
     OTHER_SAMPLE_ID
     PATHOLOGY_REPORT_FILE_NAME
     PATHOLOGY_REPORT_UUID
     SAMPLE_INITIAL_WEIGHT
     SAMPLE_TYPE
     SAMPLE_TYPE_ID
     SHORTEST_DIMENSION
     SOMATIC_STATUS
     SPECIMEN_SECOND_LONGEST_DIMENSION
     TMB_NONSYNONYMOUS
     VIAL_NUMBER


In [15]:
# === SYSTEMATYCZNY PRZEGLĄD: jakie studies GBM/LGG są dostępne w cBioPortal ===

# Mamy już studies_df z poprzedniego kroku (lista wszystkich 535 studies)
# Filtrujemy do tych, które dotyczą glejaków

# Szukamy po nazwie - "glioblastoma" lub "glioma" lub "glial" w nazwie (case-insensitive)
mask = studies_df["name"].str.contains("glio", case=False, na=False)
glioma_studies = studies_df[mask][["studyId", "name", "allSampleCount"]]

print(f"Studies dotyczące glejaków: {len(glioma_studies)}")
display(glioma_studies)

Studies dotyczące glejaków: 23


,studyId,name,allSampleCount
1,glioma_mskcc_2019,"Glioma (MSK, Clin Cancer Res 2019)",1
37,pcpg_tcga_pan_can_atlas_2018,"Pheochromocytoma and Paraganglioma (TCGA, PanC...",1
63,difg_glass_2019,"Diffuse Glioma (GLASS Consortium, Nature 2019)",1
86,difg_tcga_gdc,"Diffuse Glioma (TCGA GDC, 2025)",1
95,gbm_tcga,"Glioblastoma Multiforme (TCGA, Firehose Legacy)",1
163,glioma_msk_2018,"Glioma (MSK, Nature 2019)",1
180,pcpg_tcga,"Pheochromocytoma and Paraganglioma (TCGA, Fire...",1
188,lgg_ctf_synodos_2025,"Pediatric Low-Grade Glioma (CTF, Acta Neuropat...",1
221,gbm_tcga_pub2013,"Glioblastoma (TCGA, Cell 2013)",1
227,lgg_tcga,"Brain Lower Grade Glioma (TCGA, Firehose Legacy)",1


In [16]:
# === DIAGNOSTYKA: PanCancer Atlas studies - sprawdzamy atrybuty PATIENT ===

pancancer_studies = [
    "gbm_tcga_pan_can_atlas_2018",
    "lgg_tcga_pan_can_atlas_2018",
    "difg_tcga_gdc",        # Diffuse Glioma - może mieć już gotowe statusy
    "gbm_tcga_pub2013",     # Brennan 2013
    "gbm_tcga_pub",         # TCGA Nature 2008
]

# Słowa kluczowe szerzej - łapiemy więcej wariantów
keywords = ["MGMT", "IDH", "METHYL", "MUTATION_STATUS", "1P", "19Q", "G_CIMP", "GRADE", "SUBTYPE"]

for study_id in pancancer_studies:
    print(f"\n=== {study_id} ===")
    
    url = f"{BASE_URL}/studies/{study_id}/clinical-data"
    params = {
        "clinicalDataType": "PATIENT",
        "projection": "SUMMARY",
        "pageSize": 1_000_000,
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        df = pd.DataFrame(response.json())
        
        n_patients = df["patientId"].nunique()
        attrs = sorted(df["clinicalAttributeId"].unique())
        relevant = [a for a in attrs if any(kw in a.upper() for kw in keywords)]
        
        print(f"  Pacjentów: {n_patients}, atrybutów PATIENT łącznie: {len(attrs)}")
        if relevant:
            print(f"  ⭐ Istotne atrybuty:")
            for a in relevant:
                print(f"     {a}")
        else:
            print(f"  ❌ Brak interesujących atrybutów")
    
    except requests.exceptions.HTTPError as e:
        print(f"  ❌ Błąd HTTP: {e.response.status_code}")


=== gbm_tcga_pan_can_atlas_2018 ===
  Pacjentów: 585, atrybutów PATIENT łącznie: 34
  ⭐ Istotne atrybuty:
     SUBTYPE

=== lgg_tcga_pan_can_atlas_2018 ===
  Pacjentów: 514, atrybutów PATIENT łącznie: 34
  ⭐ Istotne atrybuty:
     SUBTYPE

=== difg_tcga_gdc ===
  Pacjentów: 516, atrybutów PATIENT łącznie: 26
  ❌ Brak interesujących atrybutów

=== gbm_tcga_pub2013 ===
  Pacjentów: 577, atrybutów PATIENT łącznie: 8
  ❌ Brak interesujących atrybutów

=== gbm_tcga_pub ===
  Pacjentów: 206, atrybutów PATIENT łącznie: 9
  ❌ Brak interesujących atrybutów


In [17]:
# === DIAGNOSTYKA: pobranie surowego pliku data_clinical_patient.txt z GitHub DataHub ===

# Bezpośredni URL do surowego pliku z repo cBioPortal/datahub
# media.githubusercontent.com obsługuje Git LFS (large file storage) - duże pliki
url = "https://media.githubusercontent.com/media/cBioPortal/datahub/master/public/gbm_tcga/data_clinical_patient.txt"

print(f"Pobieram plik z: {url}")
response = requests.get(url)
print(f"Status: {response.status_code}")
print(f"Rozmiar pliku: {len(response.text):,} znaków")
print(f"\nPierwsze 2000 znaków pliku:\n")
print(response.text[:2000])

Pobieram plik z: https://media.githubusercontent.com/media/cBioPortal/datahub/master/public/gbm_tcga/data_clinical_patient.txt
Status: 200
Rozmiar pliku: 278,508 znaków

Pierwsze 2000 znaków pliku:

#Other Patient ID	Patient Identifier	Form completion date	History lgg dx of brain tissue	Tissue Prospective Collection Indicator	Tissue Retrospective Collection Indicator	Sex	Race Category	Ethnicity Category	Prior Cancer Diagnosis Occurence	Neoadjuvant Therapy Type Administered Prior To Resection Text	Year Cancer Initial Diagnosis	First Pathologic Diagnosis Biospecimen Acquisition Method Type	First Pathologic Diagnosis Biospecimen Acquisition Other Method Type	Person Neoplasm Status	Karnofsky Performance Score	Performance Status	Performance Status Assessment Timepoint Category	Did patient start adjuvant postoperative radiotherapy?	Adjuvant Postoperative Pharmaceutical Therapy Administered Indicator	Primary Therapy Outcome Success Type	New Neoplasm Event Post Initial Therapy Indicator	Diagno